# Set up a regional MOM6 run within CESM framework (with CRR grid generation)

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [1]:
import os
from pathlib import Path
import sys

In [2]:
%load_ext autoreload
%autoreload 2
import CrocoDash as crr
import regional_mom6 as rmom6

## Step 1: Choose our domain, define workspace paths

To make sure that things are working I'd recommend starting with the default example defined below. If this runs ok, then change to a domain of your choice and hopefully it runs ok too! If not, check the [README](https://github.com/COSIMA/regional-mom6/blob/main/README.md) and [documentation](https://regional-mom6.readthedocs.io/) for troubleshooting tips.

You can log in and use [this GUI](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/download) to find the lat/lon of your domain and copy paste below.

In [5]:
expt = rmom6.create_experiment_from_config("/glade/u/home/manishrv/documents/croc/dev/configs/2n3b_angle_calc_test_nans.json",create_hgrid_and_vgrid = True)

Reading from config file....
Creating Empty Experiment Object....
Setting Default Variables.....
Creating hgrid and vgrid....
This means that some areas may only have one or two layers between the surface and sea floor. 
For increased stability, consider increasing the minimum depth, or adjusting the vertical coordinate to add more layers near the surface.
Done!


In [9]:
expt.setup_run_directory(overwrite=True)

Could not find premade run directories at  /glade/u/home/manishrv/documents/croc/regional_mom_workflows/CrocoDash/CrocoDash/rm6/regional_mom6/demos/premade_run_directories
Perhaps the package was imported directly rather than installed with conda. Checking if this is the case... Found run files. Continuing...
At least one of these is requiret to set up the experiment if you're running MOM6 standalone with the FMS coupler. 
If you're running within CESM, ignore this message.
Changed NIGLOBAL from 3040 to 1248in MOM_layout!
Changed NJGLOBAL from 3180 to 1251in MOM_layout!
Changed MASKTABLE from "mask_table.766.80x80" to  # MASKTABLE = no mask tablein MOM_layout!
Deleting indexed OBC keys from MOM_input_dict in case we have a different number of segments
Added MINIMUM_DEPTH to MOM_override with value {'value': 9.95, 'override': True, 'comment': None}
Added NK to MOM_override with value {'value': 60, 'override': True, 'comment': None}
Added OBC_NUMBER_OF_SEGMENTS to MOM_override with value

## Step 6: Modify the default input directory to make a (hopefully) runnable configuration out of the box

This step copies the default directory and modifies the `MOM_layout` files to match your experiment by inserting the right number of x, y points and CPU layout.

To run MOM6 using the [payu infrastructure](https://github.com/payu-org/payu), provide the keyword argument `using_payu = True` to the `setup_run_directory` method and an example `config.yaml` file will be appear in the run directory. The `config.yaml` file needs to be modified manually to add the locations of executables, etc.

In [5]:
expt.minimum_depth = 9.95
expt.setup_run_directory(surface_forcing = "jra", overwrite = True, with_tides = True)

Could not find premade run directories at  /glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/CrocoDash/CrocoDash/rm6/regional_mom6/demos/premade_run_directories
Perhaps the package was imported directly rather than installed with conda. Checking if this is the case... 
Found run files. Continuing...
At least one of these is requiret to set up the experiment if you're running MOM6 standalone with the FMS coupler. 
If you're running within CESM, ignore this message.
Changed NIGLOBAL from {'value': '3040', 'override': False, 'comment': '\n'} to {'value': 740, 'override': False, 'comment': '\n'} in MOM_layout!
Changed NJGLOBAL from {'value': '3180', 'override': False, 'comment': '\n'} to {'value': 780, 'override': False, 'comment': '\n'} in MOM_layout!
Changed MASKTABLE from {'value': '"mask_table.766.80x80"', 'override': False, 'comment': ' default = "MOM_mask_table"\n'} to {'value': '# MASKTABLE = no mask table', 'override': False, 'comment': ' default = "MOM_mask_table"\

In [6]:
expt.change_MOM_parameter("MAXIMUM_DEPTH",'7000', override = True) # Fred's Vertical Coordinate has a depth of 6500

Added MAXIMUM_DEPTH to MOM_override with value {'value': '7000', 'override': True, 'comment': None}


'No original val'

In [7]:
expt.expt_name

'angle_calc.3.2n3b'

# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/manishrv/documents/installs/clean_cesm3_0_alpha03a/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_nwa12 --case /glade/u/home/manishrv/cases/marbl.1--machine derecho --run-unsupported --project p93300012`

In [8]:
# Save the path where the new CESM config lives
CESM_path = Path(f"/glade/u/home/manishrv/cases/angle_calc.3.2n3b")
CESM_path

PosixPath('/glade/u/home/manishrv/cases/angle_calc.3.2n3b')

# SECTION 3: Modify the CESM run to make it regional

In [9]:
## Regional CaseGen is part of crr

reg_cas_obj = crr.regional_casegen.cesm_tools.RegionalCaseGen()


reg_cas_obj.setup_cesm(expt, CESM_path, project = "p93300012")
## OR you can explicitly state what variables are needed
# reg_cas_obj.setup_cesm_explicit(hgrid = expt.hgrid, CESMPath = CESM_path,project = "p93300612", mom_input_dir=expt.mom_input_dir, mom_run_dir=expt.mom_run_dir,date_range = expt.date_range, bathymetry_path == expt.mom_input_dir/"bathymetry.nc")

2024-12-17 16:28:33,176 - CrocoDash.regional_casegen.cesm_tools - INFO - Copying input.nml, diag_table, MOM_input_and MOM_override to /glade/u/home/manishrv/cases/angle_calc.3.2n3b/SourceMods/src.mom
2024-12-17 16:28:33,182 - CrocoDash.regional_casegen.cesm_tools - INFO - Adding NIGLOBAL = 740, NJGLOBAL = 780, and INPUTDIR = /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/angle_calc.3.2n3b to MOM_override
2024-12-17 16:28:33,182 - CrocoDash.regional_casegen.cesm_tools - INFO - Add MOM_override to parameter_filename in input.nml
2024-12-17 16:28:33,183 - CrocoDash.regional_casegen.cesm_tools - INFO - Removing reference to MOM_layout in input.nml and add MOM_override to parameter_filename in input.nml
2024-12-17 16:28:33,183 - CrocoDash.regional_casegen.cesm_tools - INFO - 
2024-12-17 16:28:33,184 - CrocoDash.regional_casegen.cesm_tools - INFO - Modifying line: parameter_filename = 'MOM_input', 'MOM_layout', 'MOM_override'
2024-12-17 16:28:33,184 - CrocoDash.regional_casegen.cesm_to

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 